# Project 2 — HuggingFace Transformers: Local NLP Playground

**Goal:** Get comfortable with **HuggingFace Transformers** — the library that lets you download and run *any* model from the HuggingFace Hub directly in Python.

**What you'll learn:**
1. What the HuggingFace Hub actually is (the "GitHub of AI models")
2. The `pipeline` abstraction — the easiest way to use *any* model
3. Three real tasks: **sentiment analysis**, **summarization**, **text generation**
4. When to reach for HuggingFace vs Ollama

---

## Ollama vs HuggingFace — the honest comparison (from Slide 22)

|  | Ollama | HuggingFace Transformers |
|---|---|---|
| **Setup** | one install, then pull | `pip install transformers torch` (heavy) |
| **Ease** | very easy | more code & config |
| **API** | built-in (`localhost:11434`) | none — you build serving |
| **Models** | curated library | **any** model on the Hub (hundreds of thousands) |
| **Best for** | building & shipping apps | research, custom pipelines, non-chat tasks |

**Rule of thumb:**
- Building a chatbot / RAG app? → **Ollama**
- Need a specific sentiment classifier, a named-entity recognizer, a translation model, or a *very specific* fine-tuned model that isn't in Ollama's library? → **HuggingFace**

---

## Step 0 — Install

> **⚠️ Warning:** `torch` is a large download (~500 MB–2 GB depending on your OS). This is one of the tradeoffs the deck warns about. Ollama gave us everything in one small install; HuggingFace needs the full PyTorch stack.

In [ ]:
%pip install -q transformers torch

## Step 1 — Sentiment Analysis (2 lines of code)

The `pipeline` function is HuggingFace's easy button. Give it a task name, and it picks a good default model, downloads it, and returns something you can call.

First time you run this, it will **download** the model (~250 MB for `distilbert`) from the HuggingFace Hub. After that, it's cached locally — no internet needed.

In [ ]:
from transformers import pipeline

# task = "sentiment-analysis" → HF picks a good default (distilbert-base-uncased-finetuned-sst-2-english)
sentiment = pipeline("sentiment-analysis")



In [ ]:
print(f"Model selected: {sentiment.model.name_or_path}")

In [ ]:
texts = [
    "I absolutely love running LLMs on my own laptop!",
    "This meeting could have been an email.",
    "The pizza was okay, nothing special.",
]

for t in texts:
    result = sentiment(t)[0]
    print(f"{result['label']:8} ({result['score']:.2f})  →  {t}")

**What just happened:** you downloaded a model from the internet, and it's now running *on your machine*. Turn off Wi-Fi and it still works. That's the same "local" story as Ollama — just via a different tool.

In [ ]:
from transformers.pipelines import get_supported_tasks

# Fetch the list of all valid task strings
available_tasks = get_supported_tasks()

# Print them out in alphabetical order for easier reading
print("Currently supported pipeline tasks:")
print("-" * 35)
for task in sorted(available_tasks):
    print(f"- {task}")

## Step 2 — Summarization

Same pattern. Different task. This is where HuggingFace really shines — you can swap between *hundreds* of pre-trained models by changing one string.

We'll use `sshleifer/distilbart-cnn-12-6` — a distilled BART model fine-tuned on CNN news articles. Roughly 300 MB.

In [ ]:
summarizer = pipeline(
    "text-generation",
    model="sshleifer/distilbart-cnn-12-6",
)

In [ ]:
article = """
Ollama is a free, open-source tool that makes running large language models locally very simple.
It handles model downloads, quantization, and serving in one package. You install it with a single
command, pull a model like Llama 3 or Mistral, and it exposes a REST API on port 11434 that your
applications can call. This means you can build AI-powered apps that cost nothing to run, work
offline, and never send user data to a third party. The main constraint is your machine's RAM —
larger models need more memory, but quantized versions of 7-8 billion parameter models fit
comfortably in 8 GB. For most learning projects and many production apps, this is the sweet spot.
"""

summary = summarizer(article, max_length=60, min_length=20, do_sample=False)
print(summary[0]['generated_text'])

## Step 3 — Feel the difference

Look at the code you wrote in **Notebook 1 (Ollama)** vs **this notebook (HuggingFace)**:

### To do "chat with a model":

**Ollama version:**
```python
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2")
llm.invoke("Hello!")
```
→ 3 lines. Small install. Works out of the box. Streaming, batching, chains all free.

**HuggingFace version:**
```python
from transformers import pipeline
generator = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")
generator([{"role": "user", "content": "Hello!"}])
```
→ 3 lines too — but under the hood, you installed PyTorch (~1 GB), you have no built-in server, and you have to think about tokenizers, chat templates, and device placement if you scale up.

### So when do you use HuggingFace?

Use HF when Ollama can't give you what you need:
- ✅ You need a **specific fine-tuned model** (e.g., a legal-domain BERT, a Hindi sentiment classifier, a medical NER model) — Ollama's library is curated and small; the Hub has *everything*.
- ✅ You need **non-chat tasks** — sentiment, classification, translation, NER, zero-shot classification, image models, audio models.
- ✅ You're doing **research** and need low-level control over tokenizers, attention, or intermediate layers.

Use Ollama when:
- ✅ You're building a **chat app / RAG / agent** — Ollama's built-in API + LangChain integration is unbeatable.
- ✅ You want the app to **just work** for anyone with a laptop, no PyTorch pain.

### One more trick from the deck (Slide 21, Path A)

If you find a great model on the HuggingFace Hub in **GGUF format**, you can run it *through Ollama*:
```bash
ollama run hf.co/{username}/{model-name-GGUF}
```
Best of both worlds — HF's model variety + Ollama's ease.

## 🎯 What you just built

You now know how to run **three different NLP tasks locally** using HuggingFace:
- Sentiment analysis
- Summarization
- Text generation

And more importantly, you know **when** to use it vs Ollama.

### Common issues
- **`Killed` / OOM** → model is bigger than your RAM. Pick a smaller model.
- **First-run is slow** → the model is downloading. It's cached after — subsequent runs are fast.
- **Where are the models stored?** → `~/.cache/huggingface/`. You can delete this folder to reclaim disk space.

### Homework 🏋️
Pick one of these HuggingFace tasks and try it yourself with `pipeline("<task>")`:
- `"translation_en_to_fr"`
- `"ner"` (named entity recognition)
- `"zero-shot-classification"`

Same 2-line pattern. Different superpower.